In [19]:
%%capture
!pip install llama-index==0.10.37 llama-index-readers-smart-pdf-loader pymupdf llamasherpa

In [32]:
import os
import sys
import getpass
import nest_asyncio
import fitz
from dotenv import load_dotenv 

nest_asyncio.apply()

load_dotenv()

sys.path.append('../helpers')

print(sys.path)


['C:\\Program Files\\WindowsApps\\PythonSoftwareFoundation.Python.3.12_3.12.2032.0_x64__qbz5n2kfra8p0\\python312.zip', 'C:\\Program Files\\WindowsApps\\PythonSoftwareFoundation.Python.3.12_3.12.2032.0_x64__qbz5n2kfra8p0\\DLLs', 'C:\\Program Files\\WindowsApps\\PythonSoftwareFoundation.Python.3.12_3.12.2032.0_x64__qbz5n2kfra8p0\\Lib', 'C:\\Program Files\\WindowsApps\\PythonSoftwareFoundation.Python.3.12_3.12.2032.0_x64__qbz5n2kfra8p0', 'c:\\Users\\maizz\\Documents\\Python Scripts\\seerah-llm\\seerahllm_env', '', 'c:\\Users\\maizz\\Documents\\Python Scripts\\seerah-llm\\seerahllm_env\\Lib\\site-packages', 'c:\\Users\\maizz\\Documents\\Python Scripts\\seerah-llm\\seerahllm_env\\Lib\\site-packages\\win32', 'c:\\Users\\maizz\\Documents\\Python Scripts\\seerah-llm\\seerahllm_env\\Lib\\site-packages\\win32\\lib', 'c:\\Users\\maizz\\Documents\\Python Scripts\\seerah-llm\\seerahllm_env\\Lib\\site-packages\\Pythonwin', '../helpers', '../helpers', '../helpers', '../helpers', '../helpers', '../hel

In [33]:
import os
import sys

# Use the absolute path to the helpers directory
helpers_path = os.path.abspath('C:\\Users\\maizz\\Documents\\Python Scripts\\seerah-llm\\helpers')
sys.path.append(helpers_path)

from text_cleaning_helpers import clean


C:\Users\maizz\Documents\Python Scripts\seerah-llm\helpers\text_cleaning_helpers.py:25: SyntaxWarning: invalid escape sequence '\*'
  "\*",
C:\Users\maizz\Documents\Python Scripts\seerah-llm\helpers\text_cleaning_helpers.py:139: SyntaxWarning: invalid escape sequence '\['
  remove_citations = lambda text: re.sub("\[\d{1,3}\]", "", text)


Data Preparation and Cleaning for RAG

In [44]:
PDF_PATH = "../data/01-sejarah-hidup-nabi-muhammad-saw.pdf"

LLMSHERPA_API_URL = "https://readers.llmsherpa.com/api/document/developer/parseDocument?renderFormat=all"

In [45]:
from llama_index.core import SimpleDirectoryReader
from llama_index.readers.file import PDFReader
from llama_index.readers.smart_pdf_loader import SmartPDFLoader 

simple_directory_reader_docs = SimpleDirectoryReader(input_files=[PDF_PATH]).load_data()

# smart_pdf_loader_docs = SmartPDFLoader(llmsherpa_api_url=LLMSHERPA_API_URL).load_data(PDF_PATH)

pdf_reader_docs = PDFReader().load_data(PDF_PATH)

In [46]:
len(simple_directory_reader_docs)

507

In [47]:
len(pdf_reader_docs)

507

In [48]:
print(pdf_reader_docs[100].get_content())

 101  menghanyutkan diri ke dalam kesenangan demikian itu. 
 
Akan tetapi jiwa Muhammad adalah jiwa yang ingin mel ihat, ingin mendengar, 
ingin mengetahui. Dan seolah tidak ikut sertanya ia bela jar seperti yang 
dilakukan teman-temannya dari anak-anak bangsawan menyeb abkan ia lebih 
keras lagi ingin memiliki pengetahuan. Karena jiwanya ya ng besar, yang 
kemudian pengaruhnya tampak berkilauan menerangi dunia,  jiwa besar yang 
selalu mendambakan kesempurnaan, itu jugalah yang menyeba bkan dia 
menjauhi foya-foya, yang biasa menjadi sasaran utama pemd uduk Mekah. Ia 
mendambakan cahaya hidup yang akan lahir dalam segala ma nifestasi 
kehidupan, dan yang akan dicapainya hanya dengan dasar keb enaran. 
Kenyataan ini dibuktikan oleh julukan yang diberikan or ang kepadanya dan 
bawaan yang ada dalam dirinya. Itu sebabnya, sejak masa ia  kanak-kanak gejala 
kesempurnaan, kedewasaan dan kejujuran hati sudah tampak,  sehingga 
penduduk Mekah semua memanggilnya Al-Amin (artinya ‘yan g dapat 

In [49]:
document = fitz.open(PDF_PATH)

def extract_text(document, opt="text"):
    '''Extract text from a page and returns a list of strings'''
    text = document.get_text(opt, sort=True) 
    text = text.split("\n")
    return text

pages = [extract_text(page) for page in document]

In [50]:
pages[42] 

['merekapun lalu menyimpang. Dan sebagian besar memang begitu. Dalam hal-',
 'hal yang berhubungan dengan sejarah Nabi kita mendapat kesempatan dalam',
 'buku           ini      mengadakan        penelitian         lebih         lanjut.',
 '',
 'Baik juga kalau dalam kesempatan  ini  kita sebutkan bahwa tugas seorang',
 'penyelidik tidak akan a priori menerima atau menolak sesuatu masalah, sebelum',
 'penelitian atau penyelidikannya  itu benar-benar meyakinkan bahwa ia sudah',
 'sepenuhnya puas dengan kenyataan yang dicapainya itu tanpa ada kekurangan.',
 'Seorang ahli sejarah dalam hal ini tidak berbeda dengan sarjana dalam ilmu',
 'pengetahuan lainnya atau dalam bidang-bidang fisika. Penulis sejarah dalam hal',
 'ini seharusnya mempelajari buku-buku  Orientalis,  juga buku-buku  sarjana-',
 'sarjana                                                                    Islam.',
 '',
 'Apabila  untuk mencapai kebenaran dan pengetahuan  itu  kita  diharuskan',
 'mengadakan  kritik dan peng

In [51]:
def get_document(file_path, pages):
    """
    Opens a PDF file and optionally selects specific pages to create a document object.

    This function utilizes the `fitz` library to open a PDF file located at `file_path`. 
    If a list of `pages` is provided, the function selects only these pages from the document.
    This is useful for focusing on certain parts of a PDF without loading the entire document into memory.

    Parameters:
        file_path (str): The path to the PDF file to be opened.
        pages (list of int, optional): A list of page numbers to select from the PDF. 
            If `None`, the entire document is loaded.

    """
    document = fitz.open(file_path)
    if pages is not None:
        document.select(pages)  # Select specific pages if pages are provided
    return document


def handle_chapter_headers_footers(strings, flag):
    """
    Modify a list of strings based on a specified flag and join them into a single string.

    This function first removes any empty strings from the input list. It then checks if the
    remaining list has more than three elements. If so, it modifies the list by removing the
    first element, last element, or both, based on the value of the flag. The final list is then
    joined into a single string with spaces separating the elements.

    Parameters:
        strings (list of str): The list of strings to modify.
        flag (str): A flag indicating the modification to perform on the list:
            - 'remove_first': Remove the first element of the list.
            - 'remove_last': Remove the last element of the list.
            - 'remove_first_last': Remove both the first and last elements of the list.
            - 'remove_first_two': Remove the first two elements of the list.
            - Any other value leaves the list unchanged.

    Returns:
        str: A single string composed of the modified list elements, separated by spaces.
    """
    # Filter out empty strings
    filtered_strings = [s for s in strings if s]
    
    # Check if the filtered list has more than three elements
    if len(filtered_strings) > 3:
        if flag == 'remove_first':
            filtered_strings = filtered_strings[1:]  # Slice off the first element
        elif flag == 'remove_last':
            filtered_strings = filtered_strings[:-1]  # Slice off the last element
        elif flag == 'remove_first_last':
            filtered_strings = filtered_strings[1:-1]  # Slice off the first and last elements
        elif flag == 'remove_first_two':
            filtered_strings = filtered_strings[2:]  # Slice off the first two elements
    
    # Join all strings with a space and return the result
    return ' '.join(filtered_strings).strip()

def extract_text(page, file_name, title, author, flag, opt="text"):
    """
    Extracts text from a specified page of a document and returns a dictionary containing
    the extracted text and associated metadata.

    The function first retrieves text from the given `page` object using the specified `opt` method.
    It then processes this text to remove chapter headers, footers, and applies various cleaning
    procedures according to the `flag` and other parameters set in the `clean` function.

    Parameters:
        page (fitz.Page): The page object from which to extract text.
        file_name (str): The name of the file from which the page is taken.
        title (str): The title of the document.
        author (str): The author of the document.
        flag (str): A flag used to customize how chapter headers and footers are handled.
        opt (str, optional): The method of text extraction to be used by `get_text`.
            Defaults to "text", but can be changed to other methods supported by the library.

    Returns:
        dict: A dictionary with two keys:
            - 'text': A string containing the cleaned and processed text from the page.
            - 'metadata': A dictionary containing metadata about the text, including the
                          page number, file name, title, and author.
    """
    
    text = page.get_text(opt, sort=True)

    text = text.split("\n")

    text = handle_chapter_headers_footers(text, flag)

    text = clean(
        text,
        extra_whitespace=True,
        broken_paragraphs=True,
        bullets=True,
        ascii=True,
        lowercase=False,
        citations=True,
        merge_split_words=True,
    )

    return {
        "text": text,
        "metadata": {
            "page_number": page.number,
            "file_name": file_name,
            "title": title,
            "author": author
        }
    }

def extract_texts_from_pdf(file_path, title, author, pages, flag):
    document = get_document(file_path, pages)
    file_name = os.path.basename(file_path)
    extracted_texts = [extract_text(page, file_path, title, author, flag) for page in document]
    return extracted_texts

In [59]:
pdf_files = [
    {
        "file_path": "../data/01-sejarah-hidup-nabi-muhammad-saw.pdf", 
        "title": "Sejarah Hidup Nabi Muhammad Saw", 
        "author": "Muhammad Husain Haikal ", 
        "pages": list(range(29, 203)),
        "flag": "remove_last"
        },
    {
        "file_path": "../data/en-concise-biography.pdf", 
        "title": "A Concise Biography of the Prophet ﷺ and His Special Traits", 
        "author": "Syeikh Haytham ibn Muhammad Sarhan", 
        "pages": list(range(2, 40)),
        "flag": "remove_last"
        },
    {
        "file_path": "../data/en-mohammad-pocket-guide.pdf", 
        "title": "Muhammad Pocket Guide", 
        "author": "Kuwait by KHT", 
        "pages": list(range(14,221)),
        "flag": "remove_first_last"
        },
    {
        "file_path": "../data/id_the_brief_of_biography_of_the_messenger.pdf", 
        "title": "Sejarah Rasulullah", 
        "author": "Al-Hafiz Abdul Ghani bin Abdul Wahid Al-Maqdisy", 
        "pages": list(range(15,45)),
        "flag": None
        },
    {
        "file_path": "../data/id-ringkasan-sirah-nabi-muhammad.pdf", 
        "title": "RINGKASAN SIRAH NABI MUHAMMAD  ﷺ",
        "author": "Prof. Dr. Muhammad bin Abdullah As-Suḥaim ", 
        "pages": list(range(1,20)),
        "flag": "remove_first_two"
        },
    {
        "file_path": "../data/ISL172%20Sirah%20The%20Biography%20of%20the%20Prophet%20Muhammad.pdf", 
        "title": "AS- SĪRAH: BIOGRAPHY OF PROPHET MUHAMMAD",  
        "author": " Dr. Shaykh Luqman Jimoh ", 
        "pages": list(range(7,125)),
        "flag": "remove_first_two"
        },
    {
        "file_path": "../data/muhammad_martin_Lings.pdf", 
        "title": "MUHAMMAD  his life based on the earliest sources",  
        "author": "Martin Lings", 
        "pages": list(range(7,258)),
        "flag": "remove_first_two"
        },
    {
        "file_path": "../data/Muhammad-the-Messenger-of-Allah.pdf", 
        "title": "Muhammad The Messenger of God",  
        "author": "Dr.Abdurrahman al-Sheha", 
        "pages": list(range(20,80)),
        "flag": None
        },
        {
        "file_path": "../data/Sirah_Nabi_Hafizh_IbnuKathir.pdf", 
        "title": "Sirah Nabi Muhammad",  
        "author": "N.Hafrzh  Ibnu Katsir", 
        "pages": list(range(20,217)),
        "flag": None
        },
        {
        "file_path": "../data/the-life-of-the-prophet-muhammad-by-leila-and-aisha.pdf", 
        "title": "The Life of the Prophet Muhammad",  
        "author": "Leila Azzam & Aisha Gouverneur", 
        "pages": list(range(20,217)),
        "flag": None
        },
]

all_texts = []

for pdf in pdf_files:
    print(f"Extracting texts from {pdf['title']} by {pdf['author']}...")
    texts = extract_texts_from_pdf(pdf["file_path"], pdf["title"], pdf["author"], pdf["pages"], pdf["flag"])
    print(f"Finished extracting texts from {pdf['title']}.")
    all_texts.extend(texts)

Extracting texts from Sejarah Hidup Nabi Muhammad Saw by Muhammad Husain Haikal ...
Finished extracting texts from Sejarah Hidup Nabi Muhammad Saw.
Extracting texts from Muhammad Pocket Guide by Kuwait by KHT...
Finished extracting texts from Muhammad Pocket Guide.
Extracting texts from MUHAMMAD  his life based on the earliest sources by Martin Lings...
Finished extracting texts from MUHAMMAD  his life based on the earliest sources.
Extracting texts from Sirah Nabi Muhammad by N.Hafrzh  Ibnu Katsir...
Finished extracting texts from Sirah Nabi Muhammad.


In [60]:
len(all_texts)

829